In [1]:
%matplotlib inline


# Sign test
In this tutorial we demonstrate how to check equal size of two samples using the sign test. To do so, we provide an example
in which we check if Heung-Min Son shoots with both feet the same number of times. 


In [2]:
import pandas as pd
import numpy as np
import json
# plotting
import matplotlib.pyplot as plt
#opening data
import os
import pathlib
import warnings

pd.options.mode.chained_assignment = None
warnings.filterwarnings('ignore')

## Opening the dataset

First we open the data. For this example we will use WyScout data from 2017/18 Premier League season.  To meet file size requirements of Github, we have to open it from different files,
but you can open the file locally from the directory you saved it in. Also, we open the file containing all players in WyScout database.



In [3]:
# open event data
file_name = 'events_England' + '.json'
path = os.path.join(str(pathlib.Path().resolve()), 'Wyscout', file_name)
with open(path) as f:
    data = json.load(f)
train = pd.DataFrame(data)

path = os.path.join(str(pathlib.Path().resolve()), 'Wyscout', 'players.json')

#open dataset with players
with open(path) as f:
    players = json.load(f)
player_df = pd.DataFrame(players)

In [6]:
print(train.info())
train.head()

<class 'pandas.DataFrame'>
RangeIndex: 643150 entries, 0 to 643149
Data columns (total 12 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   eventId       643150 non-null  int64  
 1   subEventName  643150 non-null  str    
 2   tags          643150 non-null  object 
 3   playerId      643150 non-null  int64  
 4   positions     643150 non-null  object 
 5   matchId       643150 non-null  int64  
 6   eventName     643150 non-null  str    
 7   teamId        643150 non-null  int64  
 8   matchPeriod   643150 non-null  str    
 9   eventSec      643150 non-null  float64
 10  subEventId    643150 non-null  object 
 11  id            643150 non-null  int64  
dtypes: float64(1), int64(5), object(3), str(3)
memory usage: 58.9+ MB
None


,eventId,subEventName,tags,playerId,positions,matchId,eventName,teamId,matchPeriod,eventSec,subEventId,id
0,8,Simple pass,[{'id': 1801}],25413,"[{'y': 49, 'x': 49}, {'y': 78, 'x': 31}]",2499719,Pass,1609,1H,2.758649,85,177959171
1,8,High pass,[{'id': 1801}],370224,"[{'y': 78, 'x': 31}, {'y': 75, 'x': 51}]",2499719,Pass,1609,1H,4.946850,83,177959172
2,8,Head pass,[{'id': 1801}],3319,"[{'y': 75, 'x': 51}, {'y': 71, 'x': 35}]",2499719,Pass,1609,1H,6.542188,82,177959173
3,8,Head pass,[{'id': 1801}],120339,"[{'y': 71, 'x': 35}, {'y': 95, 'x': 41}]",2499719,Pass,1609,1H,8.143395,82,177959174
4,8,Simple pass,[{'id': 1801}],167145,"[{'y': 95, 'x': 41}, {'y': 88, 'x': 72}]",2499719,Pass,1609,1H,10.302366,85,177959175


In [7]:
print(player_df.info())
player_df.head()

<class 'pandas.DataFrame'>
RangeIndex: 3603 entries, 0 to 3602
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   passportArea           3603 non-null   object
 1   weight                 3603 non-null   int64 
 2   firstName              3603 non-null   str   
 3   middleName             3603 non-null   str   
 4   lastName               3603 non-null   str   
 5   currentTeamId          3512 non-null   object
 6   birthDate              3603 non-null   str   
 7   height                 3603 non-null   int64 
 8   role                   3603 non-null   object
 9   birthArea              3603 non-null   object
 10  wyId                   3603 non-null   int64 
 11  foot                   3603 non-null   str   
 12  shortName              3603 non-null   str   
 13  currentNationalTeamId  3603 non-null   object
dtypes: int64(3), object(5), str(6)
memory usage: 394.2+ KB
None


,passportArea,weight,firstName,middleName,lastName,currentTeamId,birthDate,height,role,birthArea,wyId,foot,shortName,currentNationalTeamId
0,"{'name': 'Turkey', 'id': '792', 'alpha3code': ...",78,Harun,,Tekin,4502,1989-06-17,187,"{'code2': 'GK', 'code3': 'GKP', 'name': 'Goalk...","{'name': 'Turkey', 'id': '792', 'alpha3code': ...",32777,right,H. Tekin,4687
1,"{'name': 'Senegal', 'id': '686', 'alpha3code':...",73,Malang,,Sarr,3775,1999-01-23,182,"{'code2': 'DF', 'code3': 'DEF', 'name': 'Defen...","{'name': 'France', 'id': '250', 'alpha3code': ...",393228,left,M. Sarr,4423
2,"{'name': 'France', 'id': '250', 'alpha3code': ...",72,Over,,Mandanda,3772,1998-10-26,176,"{'code2': 'GK', 'code3': 'GKP', 'name': 'Goalk...","{'name': 'France', 'id': '250', 'alpha3code': ...",393230,,O. Mandanda,null
3,"{'name': 'Senegal', 'id': '686', 'alpha3code':...",82,Alfred John Momar,,N'Diaye,683,1990-03-06,187,"{'code2': 'MD', 'code3': 'MID', 'name': 'Midfi...","{'name': 'France', 'id': '250', 'alpha3code': ...",32793,right,A. N'Diaye,19314
4,"{'name': 'France', 'id': '250', 'alpha3code': ...",84,Ibrahima,,Konat\u00e9,2975,1999-05-25,192,"{'code2': 'DF', 'code3': 'DEF', 'name': 'Defen...","{'name': 'France', 'id': '250', 'alpha3code': ...",393247,right,I. Konat\u00e9,null


## Preparing the dataset

First, we filter the events to only keep shots. Then, we check for Son's id in the player database. As the next step,
we keep shots made by him. Then, we look for the shots made with his left (ones with *id* = 401) and right (ones with *id* = 401) foot.
In the end, we create a list with ones for shots with his left foot and -1 for shots with his right foot.



In [14]:
# take shots only
shots = train.loc[train['subEventName'] == 'Shot'] 
# look for son's id
son_id = player_df.loc[player_df["shortName"] == "Son Heung-Min"]["wyId"].iloc[0]
print( "Heung-Min Son's ID:", son_id)
# get son's shot
son_shots = shots.loc[shots["playerId"] == son_id]
print(son_shots.head())

# left leg shots
lefty_shots = son_shots.loc[son_shots.apply (lambda x:{'id':401} in x.tags, axis = 1)]
# right leg shots
righty_shots = son_shots.loc[son_shots.apply (lambda x:{'id':402} in x.tags, axis = 1)]

# create list with ones for left foot shots and -1 for right foot shots   
l = [1] * len(lefty_shots) 
l.extend([-1] * len(righty_shots))

Heung-Min Son's ID: 14911
       eventId subEventName  \
44012       10         Shot   
76728       10         Shot   
76860       10         Shot   
77625       10         Shot   
77642       10         Shot   

                                                    tags  playerId  \
44012          [{'id': 402}, {'id': 1216}, {'id': 1802}]     14911   
76728  [{'id': 1901}, {'id': 401}, {'id': 201}, {'id'...     14911   
76860  [{'id': 402}, {'id': 201}, {'id': 1216}, {'id'...     14911   
77625  [{'id': 402}, {'id': 201}, {'id': 1201}, {'id'...     14911   
77642  [{'id': 402}, {'id': 201}, {'id': 1206}, {'id'...     14911   

                                    positions  matchId eventName  teamId  \
44012  [{'y': 25, 'x': 87}, {'y': 0, 'x': 0}]  2499746      Shot    1624   
76728  [{'y': 28, 'x': 93}, {'y': 0, 'x': 0}]  2499766      Shot    1624   
76860  [{'y': 50, 'x': 79}, {'y': 0, 'x': 0}]  2499766      Shot    1624   
77625  [{'y': 40, 'x': 81}, {'y': 0, 'x': 0}]  2499766      Sh

## Testing the hypothesis

Now we can test the hypothesis that Heung-Min Son is indeed ambidextrous. To do so, a [sign test](https://en.wikipedia.org/wiki/Sign_test) is used.
We set the significance level at 0.05. After conducting the hypothesis, there's no reason to reject the null hypothesis. Therefore, we claim that
Son shoots with his right and left foot the same number of times.



In [15]:
from statsmodels.stats.descriptivestats import sign_test
test = sign_test(l, mu0 = 0)
pvalue = test[1]

if pvalue < 0.05:
    print("P-value amounts to", str(pvalue)[:5], "- We reject null hypothesis - Heung-Min Son is not ambidextrous")
else:
    print("P-value amounts to", str(pvalue)[:5], " - We do not reject null hypothesis - Heung-Min Son is ambidextrous")

P-value amounts to 0.142  - We do not reject null hypothesis - Heung-Min Son is ambidextrous
